In [4]:

#by Henry Schumacher
#-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-#
import time
start_setup = time.process_time_ns()
print('---------------------------------------')
print(time.strftime("PIXE_overview.ipynb started: %a, %d %b %Y %H:%M:%S", time.localtime()))
#-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-#
import os
import sys
import json
import uuid
import h5py
import math
import xraydb
import plotly
#-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-#
import numpy as np
import pandas as pd
# import pyxray as xy
import odrpack as odr
import seaborn as sb
#-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-#
import matplotlib.pyplot as plt
from matplotlib import ticker
from matplotlib.gridspec import GridSpec
from scipy.signal import find_peaks, argrelmax
from scipy.optimize import curve_fit
from scipy.special import voigt_profile
from getmac import get_mac_address as gma
from itertools import chain
from matplotlib.offsetbox import OffsetImage, AnnotationBbox, TextArea, VPacker
#-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-#
from colors import load_colors
from PIXE_functions import *
#-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-#

from matplotlib import rc
# rc('font',**{'family':'sans-serif','sans-serif':['Helvetica']})
## for Palatino and other serif fonts use:
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times"],
    "text.usetex": True,
    "font.size": 8,
    "pgf.rcfonts": False
})


plt.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "pgf.preamble": "\n".join([
          r'\usepackage{amsmath}',
     ]),
})

#-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-#
color_schemes = load_colors()


end_setup = time.process_time_ns()
elapsed_setup = (end_setup - start_setup)/1e6

print(f'INFO: SETUP COMPLETE ({elapsed_setup:.2f} ms)')
print('---------------------------------------')
#-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-o-#

---------------------------------------
PIXE_overview.ipynb started: Thu, 26 Mar 2026 14:49:59
INFO: SETUP COMPLETE (0.00 ms)
---------------------------------------


In [ ]:
def read_json_formatted_file(filepath, encoding="utf-8"):
    """
    Reads a file whose contents are JSON-formatted, regardless of file extension.

    Parameters:
        filepath (str): Path to the file
        encoding (str): File encoding (default: utf-8)

    Returns:
        dict or list: Parsed JSON content

    Raises:
        ValueError: If the file content is not valid JSON
        OSError: If the file cannot be read
    """
    with open(filepath, "r", encoding=encoding) as f:
        content = f.read()

    try:
        return json.loads(content)
    except json.JSONDecodeError as e:
        raise ValueError(f"File content is not valid JSON: {e}") from e
    
def file_collector(measurement:str):
    file_collection = []
    top_level_path = f'.//{measurement}'
    for file in os.listdir(top_level_path):
        if (file[-5:] == '.vspc'):
            full_path = top_level_path + '//' + file
            file_collection.append(full_path)
    return file_collection

def pixe_single_spectrum_plot_Nosave(filename:str, flag:int):
    '''
    This function will produce a simple labeled plot of uncalibrated raw-data.
    '''
    DPI = 250
    data = read_json_formatted_file(filename)
    meas_name = filename.split('//')[2].split('.')[0]
    meas_folder = filename.split('//')[1]
    
    energyPerBin = data['Calibration']['BinSize_keV/Bin'] # keV/bin
    bin_data = data['RawData'][:-1] #remove that overflow bin at position 8191
    total_counts = np.sum(bin_data)
    total_counts_incl = np.sum(data['RawData'])
    # print(bin_data)
    
    bins = np.arange(0,len(bin_data),1)
    
    scatter_color = color_schemes['c_dark']
    
    fig, ax = plt.subplots(figsize=(6,3), dpi=DPI)
    # ax.set_facecolor(color_schemes['c_back'])
    # ax.plot(bins, bin_data, lw=0.75, color=scatter_color[0], zorder=2)
    ax.step(bins, bin_data, lw=0.75, color=scatter_color[3], zorder=2)
    
    if (len(bin_data) == 8191):
        ax.set_xlim(0,8193)
        ax.set_xticks(np.arange(0,8193,1024),np.arange(0,8193,1024))
    elif (len(bin_data) == 4095):
        ax.set_xlim(0,4095)
        ax.set_xticks(np.arange(0,4097,512),np.arange(0,4097,512))
    elif (len(bin_data) == 2047):
        ax.set_xlim(0,2047)
        ax.set_xticks(np.arange(0,2049,512),np.arange(0,2049,512))
        
    ax.set_ylim(0,1.4*np.max(bin_data))
    
    if (np.max(bin_data) < 8):
        ax.set_ylim(0,8)
    elif (np.max(bin_data) == 0):
        ax.set_ylim(0,1)
        
    plt.xlabel('MCA channel')
    plt.ylabel('Counts')
    
    plt.grid(which="both")
    plt.tight_layout()

    #----------------- Information Box -----------------#
    #det_pic_file = detector_pic(measurement['det_id'])
    # img = plt.imread(det_pic_file)
    annotation = TextArea(f"X-ray measurement \n {meas_name} \n RAW DATA \n Total Counts: {total_counts}", textprops=dict(color="black", fontsize=5, multialignment='center'))
    # imagebox = OffsetImage(img, zoom=0.05)
    stacked = VPacker(children=[annotation],
                 align="center",
                 pad=0,
                 sep=5)
    
    ab = AnnotationBbox(offsetbox=stacked, xy=(0.1,0.9), xycoords='axes fraction', frameon=True)

    ax.add_artist(ab)
    #----------------- Information Box -----------------#
    plt.show()


In [ ]:
# pixe_single_spectrum_plot('.//2026_02_26//20260226-155629.vspc')

## To-Do:

#### Energy Calibration with Amersham Source Data

#### Upload to Confluence: Docu for Detector

In [ ]:
def peak_fitter(file:str, init_values:list, gauss_ident:str):
    data = read_json_formatted_file(file)
    
    bin_data = np.array(data['RawData'][:-1])
    bin_data_zerofixed = np.where(bin_data == 0, 1, bin_data)
    data_err = np.sqrt(bin_data_zerofixed)
    
    bins = np.arange(0,len(bin_data),1)
    bins_err = np.array([2]*len(bins))
    
    if (gauss_ident == 'single'):
        beta = evaluator_scipy(func=gauss_func, beta0_list=init_values,
                           x=bins, y=bin_data, xerr=bins_err, yerr=data_err)
    elif (gauss_ident == 'double'):
        beta = evaluator_scipy(func=double_gauss_func, beta0_list=init_values,
                           x=bins, y=bin_data, xerr=bins_err, yerr=data_err)
    
    return beta

### To-Do

- Presentation Flag (larger legend, ticks and labels)
- measurement name Flag
- xraydb implementation for manual search
- Double Gauss Split Flag
- Force Gauss from Init Values Flag (see a certain Rb-Measurement)
- For Spectra later, add overlapping lines where fit didn't find them

In [ ]:
def all_peaks_one_measurement(file:str, peaks:list, xlim:list, ylim:list, info:list, col_flag:bool, identifier:str, gauss_ident:list):
    
    DPI = 300
    scatter_color = color_schemes['c_dark']
    
    if (col_flag == True):
        gauss_color = color_schemes['c_light']
        gcp = [1,5]
    elif (col_flag == False):
        gauss_color = color_schemes['c_complementary']
        gcp = [0,3,4,5,6,7]
    
    data = read_json_formatted_file(file)
    
    bin_data = np.array(data['RawData'][:-1])
    bins = np.arange(0,len(bin_data),1)
    bin_data_zerofixed = np.where(bin_data == 0, 1, bin_data)
    data_err = np.sqrt(bin_data_zerofixed)
    
    meas_name = file.split('//')[2].split('.')[0]
    meas_folder = file.split('//')[1]
    
    _,ax = plt.subplots(figsize=(5,3), dpi=DPI)
    # ax.set_facecolor(color_schemes['c_back'])
    # plt.step(bins, bin_data, lw=0.75, color='black', zorder=3, label=f'{info[0]} / {meas_name}')
    plt.step(bins, bin_data, lw=0.75, color='black', zorder=3, label=f'{info[0]}')
    # plt.fill_between(bins, bin_data - data_err, bin_data + data_err, step='mid', color='black', alpha=0.3, zorder=2, label='Poissonian error')
    
    plt.xlabel('MCA channel')
    plt.ylabel('Counts')
    
    for p in range(len(peaks)):
        beta= peak_fitter(file=file, init_values=peaks[p], gauss_ident=gauss_ident[p])
        beta_export = [beta['param'],beta['errors']]
        
        
        if (gauss_ident[p] == 'single'):
            gauss_left = int(beta['param'][1]*0.95)
            gauss_right = int(beta['param'][1]*1.05)
            plt.plot(bins[gauss_left:gauss_right],gauss_func(beta['param'], bins[gauss_left:gauss_right]),lw=1, color=gauss_color[gcp[p]], zorder=3, label=fr'{info[p+1]} - line fit')
        elif (gauss_ident[p] == 'double'):
            gauss_left = int(beta['param'][1]*0.95)
            gauss_right = int(beta['param'][4]*1.05)
            plt.plot(bins[gauss_left:gauss_right],double_gauss_func(beta['param'], bins[gauss_left:gauss_right]),lw=1, color=gauss_color[gcp[p]], zorder=3, label=fr'{info[p+1]} - line fit')
            # plt.plot(bins[gauss_left:gauss_right],gauss_func(beta['param'][:3], bins[gauss_left:gauss_right]),lw=.6, color=gauss_color[gcp[p]], zorder=3, label=fr'{info[p+1]} - line fit')
            # plt.plot(bins[gauss_left:gauss_right],gauss_func(beta['param'][3:], bins[gauss_left:gauss_right]),lw=.6, color=gauss_color[gcp[p]], zorder=3, label=fr'{info[p+1]} - line fit')
            # plt.fill_between(x=bins[gauss_left:gauss_right],y1=double_gauss_func(beta['param'], bins[gauss_left:gauss_right]),alpha=0.4, color=gauss_color[gcp[p]], zorder=2, label=fr'{info[p+1]} - line fit')
        elif (gauss_ident[p] == 'single_lin'):
            gauss_left = int(beta['param'][1]*0.95)
            gauss_right = int(beta['param'][1]*1.05)
            plt.plot(bins[gauss_left:gauss_right],gauss_linear_func(beta['param'], bins[gauss_left:gauss_right]),lw=1, color=gauss_color[gcp[p]], zorder=3, label=fr'{info[p+1]} - line fit')
              
            
        csv_file_name = f'.//peak_idents{file[1:-5]}_peaks_{identifier}.csv'
        
        with open(csv_file_name, mode='a', newline='', encoding='utf-8') as csv_file:
            writer = csv.writer(csv_file)
            writer.writerows(beta_export)
        
    plt.grid(which='both')
    plt.xlim(xlim[0],xlim[1])
    if (ylim != []):
        plt.ylim(ylim[0],ylim[1])
    plt.legend(loc='best', fontsize=5)
    plt.tight_layout()
    
    plt.savefig(f'./plots/uncalibrated/{meas_folder}/{meas_name}_peaks_{identifier}.png', dpi=DPI)
    plt.savefig(f'./plots/uncalibrated/{meas_folder}/{meas_name}_peaks_{identifier}.pdf', dpi=DPI)
    
    plt.show()

In [ ]:
def grab_xrays(elem:str):
    l_fmt = '%7s  %9.1f   %8.5f  %11s'
    print('# X-ray Lines:')
    print('#  Line     Energy  Intensity       Levels')
    for key, val in xraydb.xray_lines(elem).items():
        levels = '%s-%s' % (val.initial_level, val.final_level)
        print(l_fmt % (key, val.energy, val.intensity, levels))
    return 69

In [ ]:
grab_xrays('Ag')

### To-Do
- All peaks and highlighted observed ones
- Multielemental
  - CalSam1
  - CalSam2
- CalSam2:
  - maTeck
  - chemPur

## Expected Single Element Spectrum

In [ ]:
def single_element_xray_spectrum(elem:str):
    
    E_m_1, E_m_2 = 1e8,0
    E_l_1, E_l_2 = 1e8,0
    E_k_1, E_k_2 = 1e8,0
        
    for key, val in xraydb.xray_lines(elem).items():
        if (val.energy > E_m_2 and key[0] == 'M'):
            E_m_2 = val.energy
        elif (val.energy > E_l_2 and key[0] == 'L'):
            E_l_2 = val.energy
        elif (val.energy > E_k_2 and key[0] == 'K'):
            E_k_2 = val.energy
        
        if (val.energy < E_m_1 and key[0] == 'M'):
            E_m_1 = val.energy
        elif (val.energy < E_l_1 and key[0] == 'L'):
            E_l_1 = val.energy
        elif (val.energy < E_k_1 and key[0] == 'K'):
            E_k_1 = val.energy
    
    if (E_m_1 == 1e8):
        E_m_1 = E_l_1
    elif (E_l_1 == 1e8):
        E_l_1 = E_k_1
    elif (E_k_1 == 1e8):
        E_k_1 = 1
    
    DPI = 300
    fig, ax = plt.subplots(figsize=(6,3), dpi=DPI)    
    for key, val in xraydb.xray_lines(elem).items():
        
        line_param = [val.intensity,val.energy,1]
        # print(line_param[0:2])
        line_lin = np.linspace(val.energy-200,val.energy+200,1000)
        # xlin = np.linspace(0,int(E_max),int(E_max))
        
        if key[0] == 'K':
            col = color_schemes['c_rainbow'][0]
        elif (key[0] == 'L'):
            col = color_schemes['c_rainbow'][4]
        elif (key[0] == 'M'):
            col = color_schemes['c_rainbow'][7]
            
        ax.plot(line_lin,gauss_func(line_param,line_lin), color=col, alpha=1, lw=1)
        # ax.fill_between(x=line_lin,y1=gauss_func(line_param,line_lin),alpha=val.intensity, color=col, zorder=2)#, label=fr'{key}')
   
    ax.plot([],[],color=color_schemes['c_rainbow'][0],lw=1,label='K-lines')
    ax.plot([],[],color=color_schemes['c_rainbow'][4],lw=1,label='L-lines')
    ax.plot([],[],color=color_schemes['c_rainbow'][7],lw=1,label='M-lines')
    
    ax.set_ylim(0,1.2)
    ax.set_xlim(E_m_1*0.5,E_k_2*1.1)
    
    ax.grid('both')
    ax.set_xscale('log')
    ax.set_xlabel('Energy / eV')
    ax.set_ylabel('Intensity / a.u.')
    ax.legend(loc='best')
    ax.set_title(f'{elem}')
    
    
    plt.savefig(f'./expected_spectra/mono_elemental/{elem}_ex_single_spectrum.png', transparent=False,dpi=DPI)
    plt.savefig(f'./expected_spectra/mono_elemental/{elem}_ex_single_spectrum.pdf', transparent=False,dpi=DPI)
    plt.show()
    plt.close()
        
    fig, ax = plt.subplots(figsize=(9,8), dpi=DPI, nrows=3)
    
    for key, val in xraydb.xray_lines(elem).items():
        
        line_param = [val.intensity,val.energy,1]
        # print(line_param[0:2])
        line_lin = np.linspace(val.energy-100,val.energy+100,1000)
        
        if (key[0] == 'K'):
            col = color_schemes['c_rainbow'][0]
            ax[0].plot(line_lin,gauss_func(line_param,line_lin), color=col, alpha=1, lw=0.7)
            ax[0].fill_between(x=line_lin,y1=gauss_func(line_param,line_lin),alpha=0.4, color=col, zorder=2, label=fr'{key}')
        
        elif (key[0] == 'L'):
            col = color_schemes['c_rainbow'][4]
            ax[1].plot(line_lin,gauss_func(line_param,line_lin), color=col, alpha=1, lw=0.7)
            ax[1].fill_between(x=line_lin,y1=gauss_func(line_param,line_lin),alpha=0.4, color=col, zorder=2, label=fr'{key}')
    
        elif (key[0] == 'M'):
            col = color_schemes['c_rainbow'][7]
            ax[2].plot(line_lin,gauss_func(line_param,line_lin), color=col, alpha=1, lw=0.7)
            ax[2].fill_between(x=line_lin,y1=gauss_func(line_param,line_lin),alpha=0.4, color=col, zorder=2, label=fr'{key}')
        
        if (E_m_1 == E_m_2):
            E_m_1 = 1
        elif (E_l_1 == E_l_2):
            E_l_1 = 1
        elif (E_k_1 == E_k_2):
            E_k_1 = 1
        
        ax[2].set_xlim(0,E_m_2+100)
        ax[1].set_xlim(E_l_1-100,E_l_2+100)
        ax[0].set_xlim(E_k_1-100,E_k_2+100)
        
        ax[0].set_title('K-line regime')
        ax[1].set_title('L-line regime')
        ax[2].set_title('M-line regime')
        
        
        plt.subplots_adjust(hspace=0.5)
        for i in [0,1,2]:
            ax[i].set_ylim(0,1.05)
            ax[i].set_xlabel('Energy / eV')
            ax[i].grid('both')
            ax[i].set_xscale('linear')
            ax[i].set_ylabel('Intensity / a.u.')
        
    
    plt.savefig(f'./expected_spectra/mono_elemental/{elem}_ex_single__split_spectrum.png', transparent=False,dpi=DPI)
    plt.savefig(f'./expected_spectra/mono_elemental/{elem}_ex_single_split_spectrum.pdf', transparent=False,dpi=DPI)
    plt.show()

In [ ]:
single_element_xray_spectrum('Rubidium')
single_element_xray_spectrum('Copper')
single_element_xray_spectrum('Molybdenum')
single_element_xray_spectrum('Terbium')
single_element_xray_spectrum('Silver')
single_element_xray_spectrum('Gold')
single_element_xray_spectrum('Barium')
# single_element_xray_spectrum('Indium')
# single_element_xray_spectrum('Gallium')
# single_element_xray_spectrum('Arsenic')

## Multi-Gaussian
-> into PIXE_multiGauss.ipynb

## Bremsstrahlung-Background
-> into PIXE_Bremsstrahlung.ipynb